# Predicting late deliveries
Protocol: `protocol.yaml`. Temporal holdout (`holdout.csv`), 5-split walk-forward CV on training rows, selection by mean CV ROC AUC.

**Feature decision:** `claim_paid_eur` is compensation on a claim settled *after* delivery, so it is not known at prediction time (and almost perfectly encodes the target). It is excluded. `consignment` (identifier) and `dispatched_on` (split only) are excluded per the protocol.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='859a474044a292565d72b06a6180ade1c1adbc8c72a47c4a518186876f581f27'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='fae5c02170970fe92fcadaee6d1109bb1d459c9dc033dfc487bb2c823d0f6736'
df=pd.read_csv('data.csv', parse_dates=['dispatched_on'])
hold_idx=pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==8051 and len(hold_idx)==1610
train=df.drop(index=hold_idx).sort_values('dispatched_on', kind='stable')
hold=df.loc[hold_idx]
assert train.dispatched_on.max()<=hold.dispatched_on.min()
NUM=['distance_km','weight_kg','parcels']; CAT=['carrier']
FEATURES=NUM+CAT
print(len(train), len(hold), train.late_delivery.mean(), hold.late_delivery.mean())

## Candidates (named before any scoring)

In [ ]:
def prep(): return ColumnTransformer([('n',StandardScaler(),NUM),('c',OneHotEncoder(handle_unknown='ignore'),CAT)])
candidates={
 'baseline_prior': make_pipeline(prep(), DummyClassifier(strategy='prior')),
 'logreg_C1': make_pipeline(prep(), LogisticRegression(C=1.0, max_iter=1000)),
 'logreg_C0.1': make_pipeline(prep(), LogisticRegression(C=0.1, max_iter=1000)),
 'hgb_depth3': make_pipeline(prep(), HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)),
}
print(list(candidates))

## Walk-forward CV on training rows (preprocessing is inside each pipeline, refit per fold)

In [ ]:
X=train[FEATURES]; y=train.late_delivery.to_numpy()
tscv=TimeSeriesSplit(n_splits=5)
folds=[(tr,va) for tr,va in tscv.split(X)]
cv_scores={}
for name,m in candidates.items():
    s=[]
    for tr,va in folds:
        f=clone(m).fit(X.iloc[tr],y[tr]); s.append(roc_auc_score(y[va], f.predict_proba(X.iloc[va])[:,1]))
    cv_scores[name]=s
    print(f'{name:16s} mean={np.mean(s):.4f}', np.round(s,4))
chosen=max(cv_scores,key=lambda k:np.mean(cv_scores[k])); print('chosen:',chosen)

## Refit chosen candidate on all training rows; score once on holdout

In [ ]:
final=clone(candidates[chosen]).fit(X,y)
holdout_auc=roc_auc_score(hold.late_delivery, final.predict_proba(hold[FEATURES])[:,1])
print('holdout roc_auc', round(holdout_auc,4))

In [ ]:
pos=lambda idx:[int(i) for i in idx]
evidence={
 'features':FEATURES,
 'candidates':list(candidates),
 'cv_folds':[[pos(train.index[tr]),pos(train.index[va])] for tr,va in folds],
 'cv_scores':{k:[float(v) for v in s] for k,s in cv_scores.items()},
 'chosen':chosen,
 'holdout_rows':pos(hold_idx),
 'holdout_scores':{chosen:float(holdout_auc)},
}
json.dump(evidence,open('evidence.json','w'))